In [1]:
!pip install -q dspy-ai beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.2/290.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.0 MB/s eta 0:00:00


In [2]:
import os
import re
import time
import shutil
from difflib import SequenceMatcher
from itertools import combinations
from typing import List

import pandas as pd
import requests
from bs4 import BeautifulSoup
import dspy
from pydantic import BaseModel, Field

print("Imports successful!")

Imports successful!


In [3]:
os.environ["LONGCAT_API_KEY"] = "ak_2rQ53r9IG9qZ8cE1i68i56KQ7FU6T"

lm = dspy.LM(
    "openai/LongCat-2.0",
    api_base="https://api.longcat.chat/openai/v1",
    api_key=os.environ["LONGCAT_API_KEY"],
)
dspy.configure(lm=lm)
print("LongCat-2.0 configured successfully!")

LongCat-2.0 configured successfully!


In [4]:
urls = [
    "https://en.wikipedia.org/wiki/Sustainable_agriculture",
    "https://www.nature.com/articles/d41586-025-03353-5",
    "https://www.sciencedirect.com/science/article/pii/S1043661820315152",
    "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10457221/",
    "https://www.fao.org/3/y4671e/y4671e06.htm",
    "https://www.medscape.com/viewarticle/time-reconsider-tramadol-chronic-pain-2025a1000ria",
    "https://www.sciencedirect.com/science/article/pii/S0378378220307088",
    "https://www.frontiersin.org/news/2025/09/01/rectangle-telescope-finding-habitable-planets",
    "https://www.medscape.com/viewarticle/second-dose-boosts-shingles-protection-adults-aged-65-years-2025a1000ro7",
    "https://www.theguardian.com/global-development/2025/oct/13/astro-ambassadors-stargazers-himalayas-hanle-ladakh-india",
]

print("Total URLs:", len(urls))

Total URLs: 10


In [5]:
def scrape_page(url, referer_google=False):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/139.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    }
    if referer_google:
        headers["Referer"] = "https://www.google.com/"

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        for element in soup(["script", "style", "nav", "footer", "header", "aside"]):
            element.decompose()
        return soup.get_text(separator=" ", strip=True)
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return ""


def scrape_with_fallback(url):
    text = scrape_page(url)
    if text:
        return text

    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/139.0 Safari/537.36",
            "Referer": "https://www.google.com/",
        }
        response = requests.get(url, headers=headers, timeout=30)
        soup = BeautifulSoup(response.text, "html.parser")
        title = soup.find("meta", attrs={"name": "citation_title"})
        abstract = soup.find("meta", attrs={"name": "description"})
        title_text = title.get("content", "") if title else ""
        abstract_text = abstract.get("content", "") if abstract else ""
        return f"{title_text} {abstract_text}".strip()
    except Exception as e:
        print(f"Fallback failed for {url}: {e}")
        return ""


def clean_text(text, max_chars=30000):
    text = " ".join(text.split())
    return text[:max_chars]

In [6]:
documents = []

for i, url in enumerate(urls, start=1):
    print(f"Scraping {i}/{len(urls)}...")
    text = clean_text(scrape_with_fallback(url))
    documents.append({"link": url, "text": text})
    print(f"  Characters kept: {len(text)}")
    time.sleep(1)

print("\nFinished scraping all URLs.")
print("Documents:", len(documents))

Scraping 1/10...
  Characters kept: 30000
Scraping 2/10...
  Characters kept: 4925
Scraping 3/10...
Error scraping https://www.sciencedirect.com/science/article/pii/S1043661820315152: 403 Client Error: Forbidden for url: https://www.sciencedirect.com/science/article/pii/S1043661820315152
  Characters kept: 0
Scraping 4/10...
  Characters kept: 165
Scraping 5/10...
  Characters kept: 22086
Scraping 6/10...
Error scraping https://www.medscape.com/viewarticle/time-reconsider-tramadol-chronic-pain-2025a1000ria: 403 Client Error: Forbidden for url: https://www.medscape.com/viewarticle/time-reconsider-tramadol-chronic-pain-2025a1000ria
  Characters kept: 0
Scraping 7/10...
Error scraping https://www.sciencedirect.com/science/article/pii/S0378378220307088: 403 Client Error: Forbidden for url: https://www.sciencedirect.com/science/article/pii/S0378378220307088
  Characters kept: 0
Scraping 8/10...
  Characters kept: 9313
Scraping 9/10...
Error scraping https://www.medscape.com/viewarticle/seco

In [7]:
class EntityWithAttr(BaseModel):
    entity: str = Field(description="The exact named entity or concept from the text")
    attr_type: str = Field(
        description="Semantic type such as Drug, Disease, Crop, Process, Organization, Measurement, etc."
    )


class ExtractEntities(dspy.Signature):
    """Extract important named entities and concepts from the given paragraph."""
    paragraph: str = dspy.InputField()
    entities: List[EntityWithAttr] = dspy.OutputField()


class Relationship(BaseModel):
    source: str = Field(description="Source entity")
    relation: str = Field(description="Short semantic relationship between the entities")
    target: str = Field(description="Target entity")


class ExtractRelationships(dspy.Signature):
    """Extract meaningful relationships between the provided entities."""
    paragraph: str = dspy.InputField()
    entity_list: str = dspy.InputField(description="List of allowed entities. Use only these entities.")
    relationships: List[Relationship] = dspy.OutputField()


entity_extractor = dspy.Predict(ExtractEntities)
relationship_extractor = dspy.Predict(ExtractRelationships)

print("Signatures and predictors ready!")

Signatures and predictors ready!


In [8]:
def extract_entities_dspy(text, chunk_size=8000):
    all_entities = []
    for start in range(0, len(text), chunk_size):
        chunk = text[start:start + chunk_size]
        if not chunk.strip():
            continue
        try:
            result = entity_extractor(paragraph=chunk)
            if result.entities:
                all_entities.extend(result.entities)
        except Exception as e:
            print(f"Extraction error: {e}")
    return all_entities


def normalize_entities(entities):
    unique = {}
    for item in entities:
        entity = item.entity.strip()
        attr_type = item.attr_type.strip()
        if not entity:
            continue
        key = entity.lower()
        if key not in unique:
            unique[key] = {"entity": entity, "attr_type": attr_type}
    return list(unique.values())


def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def deduplicate_entity_names(names, threshold=0.85):
    unique = []
    for name in names:
        name = name.strip()
        if not name:
            continue
        if not any(similarity(name, existing) >= threshold for existing in unique):
            unique.append(name)
    return unique

In [9]:
def extract_relationships_dspy(text, entities):
    entity_names = [e["entity"] for e in entities]
    entity_text = ", ".join(entity_names)

    try:
        result = relationship_extractor(paragraph=text[:8000], entity_list=entity_text)
        relationships = []
        for r in result.relationships:
            if r.source in entity_names and r.target in entity_names and r.source != r.target:
                relationships.append({
                    "source": r.source,
                    "relation": r.relation[:40].strip(),
                    "target": r.target,
                })
        return relationships
    except Exception as e:
        print("Relationship extraction error:", e)
        return []

In [10]:
def clean_mermaid_text(text):
    text = str(text).replace('"', "'")
    text = " ".join(text.split())
    return text[:40]


def triples_to_mermaid(triples, entity_list):
    valid_entities = set(entity_list)
    lines = ["graph LR"]

    entity_to_id = {entity: f"N{i}" for i, entity in enumerate(entity_list)}
    for entity, node_id in entity_to_id.items():
        safe_entity = entity.replace('"', "'")
        lines.append(f'    {node_id}["{safe_entity}"]')

    for triple in triples:
        source, target = triple["source"], triple["target"]
        if source not in valid_entities or target not in valid_entities:
            continue
        relation = clean_mermaid_text(triple["relation"])
        lines.append(f'    {entity_to_id[source]} -->|"{relation}"| {entity_to_id[target]}')

    return "\n".join(lines)

In [11]:
def process_document(link, text):
    print(f"\nProcessing: {link}")

    entities = normalize_entities(extract_entities_dspy(text))
    print(f"Entities extracted: {len(entities)}")

    unique_names = deduplicate_entity_names([e["entity"] for e in entities])
    unique_entities = []
    for name in unique_names:
        for entity in entities:
            if entity["entity"].lower() == name.lower():
                unique_entities.append(entity)
                break
    print(f"After deduplication: {len(unique_entities)}")

    relationships = extract_relationships_dspy(text, unique_entities)
    print(f"Relationships: {len(relationships)}")

    mermaid = triples_to_mermaid(relationships, [e["entity"] for e in unique_entities])

    return {"link": link, "entities": unique_entities, "relationships": relationships, "mermaid": mermaid}


def save_result(result, index, out_dir="final_output"):
    mermaid_path = f"{out_dir}/mermaid/mermaid_{index}.md"
    with open(mermaid_path, "w", encoding="utf-8") as f:
        f.write("```mermaid\n")
        f.write(result["mermaid"])
        f.write("\n```\n")

    rows = []
    for entity in result["entities"]:
        rows.append({"link": result["link"], "tag": entity["entity"], "tag_type": entity["attr_type"]})
    return rows


def run_pipeline(documents, out_dir="final_output"):
    all_rows = []
    os.makedirs(f"{out_dir}/mermaid", exist_ok=True)

    for i, doc in enumerate(documents, start=1):
        if not doc["text"].strip():
            print(f"Skipping URL {i}: no text available")
            continue
        result = process_document(doc["link"], doc["text"])
        all_rows.extend(save_result(result, i, out_dir))

    final_df = pd.DataFrame(all_rows, columns=["link", "tag", "tag_type"])
    final_df = final_df.drop_duplicates(subset=["link", "tag"]).reset_index(drop=True)
    final_df.to_csv(f"{out_dir}/tags.csv", index=False)
    return final_df

In [12]:
final_df = run_pipeline(documents)

print("\nFinal CSV structure:")
print(final_df.head())
print("\nTotal rows:", len(final_df))


Processing: https://en.wikipedia.org/wiki/Sustainable_agriculture
Extraction error: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
Extraction error: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
Extraction error: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
Extraction error: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
Entities extracted: 0
After deduplication: 0
Relationship extraction error: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
Relationships: 0

Processing: https://www.nature.c

In [13]:
def validate_output(out_dir="final_output"):
    print("========== VALIDATION ==========\n")
    csv_path = f"{out_dir}/tags.csv"
    if os.path.exists(csv_path):
        check_df = pd.read_csv(csv_path)
        print("Ok tags.csv exists")
        print("Ok Columns:", list(check_df.columns))
        print("Ok Duplicate link+tag:", check_df.duplicated(subset=["link", "tag"]).sum())
    else:
        print("X tags.csv missing")

    mermaid_dir = f"{out_dir}/mermaid"
    if os.path.exists(mermaid_dir):
        mermaid_files = sorted(f for f in os.listdir(mermaid_dir) if f.endswith(".md"))
        print(f"Ok Mermaid files found: {len(mermaid_files)}")
        for filename in mermaid_files:
            with open(os.path.join(mermaid_dir, filename), "r", encoding="utf-8") as f:
                content = f.read()
            print(f"  {'Ok' if 'graph LR' in content else 'X'} {filename}")
    print("\n========== END ==========")


validate_output()

========== VALIDATION ==========

Ok tags.csv exists
Ok Columns: ['link', 'tag', 'tag_type']
Ok Duplicate link+tag: 0
Ok Mermaid files found: 6
  Ok mermaid_1.md
  Ok mermaid_10.md
  Ok mermaid_2.md
  Ok mermaid_4.md
  Ok mermaid_5.md
  Ok mermaid_8.md

========== END ==========


In [14]:
README_TEXT = """
# Clarion Services - DSPy Practical Assignment

## Project
Structuring Unstructured Data with LLMs

## Pipeline

Web URLs
-> Web Scraping
-> Text Cleaning
-> DSPy Entity Extraction
-> Pydantic Structured Output
-> Entity Deduplication
-> Relationship Extraction
-> Mermaid Knowledge Graphs
-> Structured CSV

## Outputs

- tags.csv
- mermaid_1.md to mermaid_10.md

## Technologies

- Python
- DSPy
- Pydantic
- BeautifulSoup
- Pandas
- LongCat LLM
- Mermaid

## Note

The LongCat API is used for the LLM-based entity and relationship
extraction stages.
"""

with open("final_output/README.md", "w", encoding="utf-8") as f:
    f.write(README_TEXT)

shutil.make_archive("Clarion_assignment", "zip", "final_output")
print("Created: Clarion_assignment.zip")

Created: Clarion_assignment.zip
